<a href="https://colab.research.google.com/github/AlfredaDhaifullah04/PCVK_Ganjil_2026/blob/main/PCVK_M9_W9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum Modul 9: Morfologi Citra

Notebook ini berisi implementasi kode praktikum dan tugas dari Modul 9 mengenai operasi Morfologi (Erosi, Dilasi, Opening, Closing, Top-hat, Bottom-hat, Skeleton, dan Thickening).

In [ ]:
import cv2 as cv2
import numpy as np
from matplotlib import pyplot as plt
from skimage import io, filters, morphology, img_as_float, color
import urllib.request

# Fungsi pembantu untuk mengunduh gambar sampel dari internet jika tidak ada di GDrive
def download_image(url, filename):
    urllib.request.urlretrieve(url, filename)
    return filename

# Mengunduh gambar sampel (plat nomor, anak kucing, dan bunga)
plat_url = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/licenseplate_motion.jpg'
kitten_url = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/messi5.jpg' # alternatif
lily_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/a/a4/Lilium_candidum_1.jpg/600px-Lilium_candidum_1.jpg'

plat_img_path = download_image(plat_url, 'plat_nomer.jpg')
kitten_img_path = download_image(kitten_url, 'kitten01.jpg')
lily_img_path = download_image(lily_url, 'lily.jpg')


## 3. Dilasi Tanpa Library Morphology OpenCV (Cross 3x3)

In [ ]:
def dilasi_citra(F, w):
    p, q = F.shape
    imgD = np.zeros((p, q), dtype=np.uint8)
    # Generate SED
    SED = np.ones((w, w), dtype=np.uint8)
    constant1 = (w - 1) // 2

    # Proses Dilasi
    for i in range(constant1, p - constant1):
        for j in range(constant1, q - constant1):
            temp = F[i - constant1:i + constant1 + 1, j - constant1:j + constant1 + 1]
            product = temp * SED
            imgD[i, j] = np.max(product)
    return imgD

img_d = cv2.imread(plat_img_path, 0)
ret, thresh = cv2.threshold(img_d, 125, 200, cv2.THRESH_BINARY)
w = 5
imgD = dilasi_citra(thresh, w)

plt.figure(figsize=(10, 5))
plt.subplot(121), plt.imshow(img_d, cmap='gray')
plt.title('Citra Awal'), plt.xticks([]), plt.yticks([])
plt.subplot(122), plt.imshow(imgD, cmap='gray')
plt.title('Citra Hasil Dilasi (Tanpa Library)'), plt.xticks([]), plt.yticks([])
plt.show()

## 4. Dilasi Menggunakan Library Morphology OpenCV

In [ ]:
img = cv2.imread(plat_img_path)
ret, thresh = cv2.threshold(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), 127, 225, cv2.THRESH_BINARY)
kernel = np.ones((3, 3), np.uint8)
dilasi = cv2.dilate(thresh, kernel, iterations=1)

plt.figure(figsize=(10, 5))
plt.subplot(121), plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)), plt.title('Citra Awal')
plt.xticks([]), plt.yticks([])
plt.subplot(122), plt.imshow(dilasi, cmap='gray'), plt.title('Citra Hasil Dilasi (Library)')
plt.xticks([]), plt.yticks([])
plt.show()

## 5. Erosi Tanpa Library Morphology OpenCV (Square 5x5)

In [ ]:
def erosi_citra(F, k):
    m, n = F.shape
    SE = np.ones((k, k), dtype=np.uint8)
    constant = (k - 1) // 2
    imgE = np.zeros((m, n), dtype=np.uint8)

    for i in range(constant, m - constant):
        for j in range(constant, n - constant):
            temp = F[i - constant:i + constant + 1, j - constant:j + constant + 1]
            product = temp * SE
            imgE[i, j] = np.min(product)
    return imgE

img_e = cv2.imread(plat_img_path, 0)
ret1, thresh1 = cv2.threshold(img_e, 175, 225, cv2.THRESH_BINARY)
k = 5
imgE = erosi_citra(thresh1, k)

plt.figure(figsize=(10, 5))
plt.subplot(121), plt.imshow(img_e, cmap='gray')
plt.title('Citra Awal'), plt.xticks([]), plt.yticks([])
plt.subplot(122), plt.imshow(imgE, cmap='gray')
plt.title('Citra Hasil Erosi (Tanpa Library)'), plt.xticks([]), plt.yticks([])
plt.show()

## 6. Erosi Menggunakan Library Morphology OpenCV

In [ ]:
img = cv2.imread(plat_img_path, 0)
kernel = np.ones((5, 5), np.uint8)
erosion = cv2.erode(img, kernel, iterations=1)

plt.figure(figsize=(10, 5))
plt.subplot(121), plt.imshow(img, cmap='gray'), plt.title('Citra Awal')
plt.xticks([]), plt.yticks([])
plt.subplot(122), plt.imshow(erosion, cmap='gray'), plt.title('Citra Hasil Erosi (Library)')
plt.xticks([]), plt.yticks([])
plt.show()

## 7. Operasi Opening (Tanpa dan Dengan Library)

In [ ]:
# Menggunakan fungsi dari modul sebelumnya
img_o = cv2.imread(plat_img_path, 0)
ret0, thresh0 = cv2.threshold(img_o, 127, 225, cv2.THRESH_BINARY)
k = 7

# Opening = Erosi lalu Dilasi (Tanpa Library)
imgO_manual = dilasi_citra(erosi_citra(thresh0, k), k)

# Opening Dengan Library
kernel_op = np.ones((3, 3), np.uint8)
openn = cv2.morphologyEx(thresh0, cv2.MORPH_OPEN, kernel_op)

plt.figure(figsize=(15, 5))
plt.subplot(131), plt.imshow(img_o, cmap='gray')
plt.title('Citra Awal'), plt.xticks([]), plt.yticks([])
plt.subplot(132), plt.imshow(imgO_manual, cmap='gray')
plt.title('Opening (Manual)'), plt.xticks([]), plt.yticks([])
plt.subplot(133), plt.imshow(openn, cmap='gray')
plt.title('Opening (Library)'), plt.xticks([]), plt.yticks([])
plt.show()

## 8. Operasi Closing (Tanpa dan Dengan Library)

In [ ]:
img_c = cv2.imread(plat_img_path, 0)
retc, threshc = cv2.threshold(img_c, 127, 225, cv2.THRESH_BINARY)
k = 7

# Closing = Dilasi lalu Erosi (Tanpa Library)
imgC_manual = erosi_citra(dilasi_citra(threshc, k), k)

# Closing Dengan Library
kernel_cl = np.ones((3, 3), np.uint8)
closs = cv2.morphologyEx(threshc, cv2.MORPH_CLOSE, kernel_cl)

plt.figure(figsize=(15, 5))
plt.subplot(131), plt.imshow(img_c, cmap='gray')
plt.title('Citra Awal'), plt.xticks([]), plt.yticks([])
plt.subplot(132), plt.imshow(imgC_manual, cmap='gray')
plt.title('Closing (Manual)'), plt.xticks([]), plt.yticks([])
plt.subplot(133), plt.imshow(closs, cmap='gray')
plt.title('Closing (Library)'), plt.xticks([]), plt.yticks([])
plt.show()

## 9. Dilasi Cross 3x3 dan 5x5

In [ ]:
def dilasi_cross(F, SED):
    p, q = F.shape
    imgD = np.zeros((p, q), dtype=np.uint8)
    w = SED.shape[0]
    constant1 = (w - 1) // 2
    for i in range(constant1, p - constant1):
        for j in range(constant1, q - constant1):
            temp = F[i - constant1:i + constant1 + 1, j - constant1:j + constant1 + 1]
            product = temp * SED
            imgD[i, j] = np.max(product)
    return imgD

img_d = cv2.imread(plat_img_path, 0)
ret, thresh = cv2.threshold(img_d, 127, 225, cv2.THRESH_BINARY)

SED_cross3 = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], dtype=np.uint8)
imgD_cross3 = dilasi_cross(thresh, SED_cross3)

SED_cross5 = np.array([
    [0, 0, 1, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 1, 1, 1, 1],
    [0, 0, 1, 0, 0],
    [0, 0, 1, 0, 0]
], dtype=np.uint8)
imgD_cross5 = dilasi_cross(thresh, SED_cross5)

plt.figure(figsize=(15, 5))
plt.subplot(131), plt.imshow(img_d, cmap='gray'), plt.title('Citra Awal')
plt.subplot(132), plt.imshow(imgD_cross3, cmap='gray'), plt.title('Dilasi Cross 3x3')
plt.subplot(133), plt.imshow(imgD_cross5, cmap='gray'), plt.title('Dilasi Cross 5x5')
plt.show()

## 13. Top-Hat

In [ ]:
filterSize = (3, 3)
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, filterSize)
img = cv2.imread(plat_img_path)
img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

tophat_img = cv2.morphologyEx(img_gray, cv2.MORPH_TOPHAT, kernel)

plt.figure(figsize=(10, 5))
plt.subplot(121), plt.imshow(img_gray, cmap='gray'), plt.title('Citra Awal')
plt.subplot(122), plt.imshow(tophat_img, cmap='gray'), plt.title('Citra Hasil Top-Hat')
plt.show()

## 14. Black-Hat (Bottom-Hat)

In [ ]:
Kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (13, 5))
img_k = cv2.imread(kitten_img_path)
img_k_gray = cv2.cvtColor(img_k, cv2.COLOR_BGR2GRAY)

blackhat = cv2.morphologyEx(img_k_gray, cv2.MORPH_BLACKHAT, Kernel)

plt.figure(figsize=(10, 5))
plt.subplot(121), plt.imshow(img_k_gray, cmap='gray'), plt.title('Citra Awal')
plt.subplot(122), plt.imshow(blackhat, cmap='gray'), plt.title('Citra Hasil Black-Hat')
plt.show()

## 15 & 16. Skeletonize dan Skeleton Inverse

In [ ]:
from skimage.morphology import skeletonize

img_l = cv2.imread(lily_img_path, 0)
# Thresholding skimage
binary = img_l > filters.threshold_triangle(img_l)
binary_cp = binary.copy()

skeleton = skeletonize(binary_cp)

plt.figure(figsize=(15, 5))
plt.subplot(131), plt.imshow(img_l, cmap='gray'), plt.title('Original Image')
plt.subplot(132), plt.imshow(skeleton, cmap='gray'), plt.title('Skeleton Image')

# Inverse Skeleton (Background Hitam menjadi target)
binary_inv = img_l < filters.threshold_triangle(img_l)
skeleton_inv = skeletonize(binary_inv)
plt.subplot(133), plt.imshow(skeleton_inv, cmap='gray_r'), plt.title('Skeleton (Inverse)')
plt.show()

## 17. Thickening / Thinning

In [ ]:
image = img_as_float(color.rgb2gray(io.imread(lily_img_path)))
image_binary = image < 0.5

out_thin = morphology.thin(image_binary)

f, (ax0, ax1) = plt.subplots(1, 2, figsize=(10, 5))
ax0.imshow(image, cmap='gray')
ax0.set_title('Input')
ax1.imshow(out_thin, cmap='gray')
ax1.set_title('Thin')
plt.show()

## E. Tugas Praktikum (Penerapan pada Citra KTP)
Pada bagian ini kita mensimulasikan penggunaan morfologi matematis untuk membantu membaca dan memperjelas teks pada citra menyerupai KTP (sebagai contoh, kita menggunakan citra sample yang menyerupai KTP atau teks plat nomor).

In [ ]:
# Simulasi Pra-pemrosesan KTP dengan Morfologi
# Kita gunakan operasi Top-hat dan Closing untuk menonjolkan fitur teks.

img_ktp_sim = cv2.imread(plat_img_path, 0) # Menggunakan plat nomor sebagai representasi teks bising

# 1. Thresholding
ret, binary_ktp = cv2.threshold(img_ktp_sim, 127, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)

# 2. Operasi Closing untuk menyambungkan teks yang terputus
kernel_rect = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
ktp_closed = cv2.morphologyEx(binary_ktp, cv2.MORPH_CLOSE, kernel_rect)

plt.figure(figsize=(15, 5))
plt.subplot(131), plt.imshow(img_ktp_sim, cmap='gray'), plt.title('Simulasi KTP Awal')
plt.subplot(132), plt.imshow(binary_ktp, cmap='gray'), plt.title('Binarisasi')
plt.subplot(133), plt.imshow(ktp_closed, cmap='gray'), plt.title('Morfologi Closing (Teks Lebih Jelas)')
plt.show()